# Demo nhỏ — OCR.space trên một keyframe

Notebook này dùng **một request** để kiểm tra API key, văn bản nhận dạng và bounding box trước khi chạy notebook OCR toàn bộ keyframes.

In [ ]:
# Không nâng cấp Pillow/Matplotlib trong runtime đang chạy: có thể làm lệch các module PIL đã nạp.
!pip -q install requests python-dotenv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cấu hình

Chép `.env` vào `/content/drive/MyDrive/AI Challenge/.env`. File phải có dòng `OCR_SPACE_KEY=...`. Sửa `IMAGE_PATH` thành một keyframe có thật trên Drive.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = '/content/drive/MyDrive/AI Challenge/.env'  #@param {type:'string'}
IMAGE_PATH = '/content/drive/MyDrive/AI Challenge/Dataset_Directory/Keyframes_L21/keyframes/L21_V001/001.jpg'  #@param {type:'string'}
OCR_LANGUAGE = 'auto'  #@param {type:'string'}
OCR_ENGINE = 2  #@param {type:'integer'}
TEST_IMAGE_COUNT = 10  #@param {type:'integer'}
LANGUAGE_ALIASES = {'vi': 'vnm', 'vn': 'vnm', 'vie': 'vnm', 'vietnamese': 'vnm'}
OCR_LANGUAGE = LANGUAGE_ALIASES.get(OCR_LANGUAGE.strip().lower(), OCR_LANGUAGE.strip().lower())

env_path = Path(ENV_FILE)
image_path = Path(IMAGE_PATH)
assert env_path.is_file(), f'Không tìm thấy .env: {env_path}'
assert image_path.is_file(), f'Không tìm thấy ảnh: {image_path}'
load_dotenv(env_path, override=False)
raw_keys = os.getenv('OCR_SPACE_KEYS', '').strip() or os.getenv('OCR_SPACE_KEY', '').strip()
def parse_api_keys(raw):
    if not raw: return []
    try:
        parsed = json.loads(raw)
        values = parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        values = raw.replace(';', ',').split(',')
    return list(dict.fromkeys(str(value).strip() for value in values if str(value).strip()))
api_keys = parse_api_keys(raw_keys)
assert api_keys, 'Thiếu OCR_SPACE_KEY hoặc OCR_SPACE_KEYS trong .env'
all_images = sorted(
    (p for p in image_path.parent.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}),
    key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else 0, p.name),
)
assert all_images, f'Không có ảnh trong {image_path.parent}'
sample_count = min(max(1, TEST_IMAGE_COUNT), len(all_images))
sample_indexes = [round(i * (len(all_images) - 1) / max(1, sample_count - 1)) for i in range(sample_count)]
image_paths = [all_images[i] for i in sample_indexes]
print(f'Chọn {len(image_paths)}/{len(all_images)} ảnh rải đều trong:', image_path.parent)
print('Tên ảnh:', ', '.join(p.name for p in image_paths))
print(f'Đã nạp {len(api_keys)} API key (không hiển thị giá trị)')

## Gọi OCR.space API

In [ ]:
import requests

API_URL = 'https://api.ocr.space/parse/image'
form = {
            'language': OCR_LANGUAGE,
            'OCREngine': str(OCR_ENGINE),
            'isOverlayRequired': 'true',
            'detectOrientation': 'true',
            'scale': 'true',
}
active_key_index = 0
def ocr_one(image_path):
    global active_key_index
    last_error = None
    request_configs = [form]
    if OCR_LANGUAGE != 'auto':
        request_configs.append({**form, 'language': 'auto', 'OCREngine': '2'})
    for request_form in request_configs:
        for offset in range(len(api_keys)):
            key_index = (active_key_index + offset) % len(api_keys)
            try:
                with image_path.open('rb') as image_file:
                    response = requests.post(
                        API_URL, headers={'apikey': api_keys[key_index]}, data=request_form,
                        files={'file': (image_path.name, image_file, 'application/octet-stream')}, timeout=90,
                    )
                response.raise_for_status()
                result = response.json()
                if result.get('IsErroredOnProcessing'):
                    raise RuntimeError(result.get('ErrorMessage') or result.get('ErrorDetails'))
                pages = result.get('ParsedResults') or []
                if not pages: raise RuntimeError('API không trả ParsedResults')
                active_key_index = key_index
                text = '\n'.join((page.get('ParsedText') or '').strip() for page in pages).strip()
                return {'path': image_path, 'result': result, 'pages': pages, 'text': text, 'key_index': key_index}
            except Exception as exc:
                last_error = exc
                if 'E201' in str(exc): break
                print(f'  key #{key_index + 1} lỗi: {exc}')
    raise RuntimeError(f'OCR thất bại cho {image_path.name}') from last_error

records = []
for index, path in enumerate(image_paths, 1):
    record = ocr_one(path)
    records.append(record)
    ms = record['result'].get('ProcessingTimeInMilliseconds')
    print(f"[{index}/{len(image_paths)}] {path.name}: {ms} ms, key #{record['key_index'] + 1} -> {record['text'][:100].replace(chr(10), ' | ') or '(không có chữ)'}")

## Hiển thị bounding box

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont

def extract_detections(pages):
  detections = []
  for page in pages:
    overlay = page.get('TextOverlay') or {}
    for line in overlay.get('Lines') or []:
        for word in line.get('Words') or []:
            label = (word.get('WordText') or '').strip()
            left, top = int(word.get('Left', 0)), int(word.get('Top', 0))
            width, height = int(word.get('Width', 0)), int(word.get('Height', 0))
            if label and width > 0 and height > 0:
                detections.append({
                    'text': label,
                    'box': [left, top, left + width, top + height],
                })
  return detections

def annotate(record):
  image = Image.open(record['path']).convert('RGB')
  detections = extract_detections(record['pages'])
  annotated = image.copy()
  draw = ImageDraw.Draw(annotated)
  font_path = font_manager.findfont('DejaVu Sans')
  font = ImageFont.truetype(font_path, max(13, image.height // 45))
  line_width = max(2, image.height // 400)
  for index, detection in enumerate(detections, 1):
    left, top, right, bottom = detection['box']
    color = (0, 255, 0)
    draw.rectangle([left, top, right, bottom], outline=color, width=line_width)
    label = f"{index}: {detection['text']}"
    tx1, ty1, tx2, ty2 = draw.textbbox((0, 0), label, font=font)
    label_width, label_height = tx2 - tx1, ty2 - ty1
    label_y = top - label_height - 3
    if label_y < 0:
        label_y = min(bottom + 2, image.height - label_height - 3)
    label_x = min(left, max(0, image.width - label_width - 5))
    draw.rectangle(
        [label_x, label_y, label_x + label_width + 4, label_y + label_height + 3],
        fill=color,
    )
    draw.text((label_x + 2, label_y + 1), label, fill=(0, 0, 0), font=font)
  return annotated, detections

ANNOTATED_DIR = Path('/content/ocr_space_demo/annotated')
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
for index, record in enumerate(records, 1):
    annotated, detections = annotate(record)
    record['detections'] = detections
    annotated_path = ANNOTATED_DIR / f"{record['path'].stem}_ocrspace.jpg"
    annotated.save(annotated_path, quality=92)
    record['annotated_path'] = annotated_path
    plt.figure(figsize=(16, 16 * annotated.height / annotated.width))
    plt.imshow(annotated)
    plt.title(f"[{index}/{len(records)}] {record['path'].name} — {len(detections)} word boxes")
    plt.axis('off'); plt.tight_layout(); plt.show()
    print('Text:', record['text'] or '(rỗng)')
    print('Đã lưu:', annotated_path, '\n')
print('Tải về: Files → ocr_space_demo → annotated → chuột phải → Download')

## Xem JSON rút gọn

Không hiển thị API key. Cell này giúp kiểm tra cấu trúc dữ liệu trước khi tích hợp.

In [ ]:
import json
summary = {
    'image_count': len(records),
    'language': OCR_LANGUAGE,
    'engine': OCR_ENGINE,
    'results': [{
        'image': record['path'].name, 'text': record['text'],
        'word_count': len(record['detections']), 'detections': record['detections'],
        'processing_ms': record['result'].get('ProcessingTimeInMilliseconds'),
    } for record in records],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))